# Full benchmark run: 50 prompts × 3 mechanism variants → handoff CSVs

**This is the artifact Wes and Alex/Swetha consume.** It runs the full benchmark and dumps three CSVs into `results/`:

- `results/defended.csv` — welfare-priced reserve auction
- `results/revmax.csv` — VCG with no welfare reserve (revenue-maximizing baseline)
- `results/pure_relevance.csv` — pick the most-relevant ad, no auction (relevance baseline)

Each CSV has one row per prompt with the `AdServeRecord` schema (see `auction/types.py`).

**Cost:** zero LLM API calls. Pure local compute over ~1–2 minutes after embeddings warm up.

## Setup

In [1]:
# If on Colab, mount Drive and cd into the project. If local, skip.
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    %cd '/content/drive/Shared drives/[OIT 277] Final project/Eric code'
    import os
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except ImportError:
    pass  # local

Mounted at /content/drive
/content/drive/Shared drives/[OIT 277] Final project/Eric code


In [2]:
%pip install -q -r ../requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'


In [3]:
import sys, json
from pathlib import Path
from dataclasses import asdict
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, '..')
from auction import benchmark, data_pipeline, mechanism
from auction.types import AdServeRecord
import config

## 1. Load benchmark + products + index

In [4]:
prompts = benchmark.load_seed_prompts()
products = benchmark.load_amazon_products_cached(n=10000)
product_index = data_pipeline.build_product_index(products)
advertisers = mechanism.generate_advertisers(products, n=len(products), seed=42)
print(f'{len(prompts)} prompts | {len(products)} products | {len(advertisers)} advertisers')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

50 prompts | 10000 products | 10000 advertisers


## 2. Run all 50 prompts × 3 mechanism variants

Uses `mechanism.serve_ad()` — the high-level entry that wraps retrieve → auction → rewrite into one call. Output is one `AdServeRecord` per (prompt, variant), 150 records total.

In [5]:
VARIANTS = ['defended', 'revmax', 'pure_relevance']
RETRIEVAL_K = 20  # sponsored-search-like depth

all_records: dict[str, list[AdServeRecord]] = {v: [] for v in VARIANTS}

for variant in VARIANTS:
    for p in tqdm(prompts, desc=f'{variant:>14}'):
        record = mechanism.serve_ad(
            prompt=p['prompt'],
            products=products,
            product_index=product_index,
            advertisers=advertisers,
            variant=variant,
            clean_answer=p.get('clean_answer', ''),
            prompt_id=p['id'],
            is_sensitive=p.get('is_sensitive', False),
            is_borderline=p.get('is_borderline', False),
            category=p.get('category', ''),
            k=RETRIEVAL_K,
            seed=0,
        )
        all_records[variant].append(record)

print(f'Done. {sum(len(v) for v in all_records.values())} records across {len(VARIANTS)} variants.')

      defended:   0%|          | 0/50 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


        revmax:   0%|          | 0/50 [00:00<?, ?it/s]

pure_relevance:   0%|          | 0/50 [00:00<?, ?it/s]

Done. 150 records across 3 variants.


## 3. Save the three handoff CSVs

In [6]:
results_dir = Path('../results')
results_dir.mkdir(exist_ok=True)

def records_to_df(records: list[AdServeRecord]) -> pd.DataFrame:
    rows = [asdict(r) for r in records]
    # JSON-encode list field so CSV stays readable
    for row in rows:
        row['candidate_product_ids'] = json.dumps(row['candidate_product_ids'])
    return pd.DataFrame(rows)

for variant, records in all_records.items():
    df = records_to_df(records)
    out_path = results_dir / f'{variant}.csv'
    df.to_csv(out_path, index=False)
    print(f'  wrote {out_path}  ({len(df)} rows × {len(df.columns)} cols)')

  wrote ../results/defended.csv  (50 rows × 22 cols)
  wrote ../results/revmax.csv  (50 rows × 22 cols)
  wrote ../results/pure_relevance.csv  (50 rows × 22 cols)


## 4. Sanity-check the outputs

Per-variant counts of: ads served, ads gated, average clearing price.

In [7]:
summary = []
for variant in VARIANTS:
    df = records_to_df(all_records[variant])
    summary.append({
        'variant': variant,
        'n_total': len(df),
        'n_served': df['winner_product_id'].notna().sum(),
        'n_gated_sensitive': ((df['is_sensitive']) & (df['winner_product_id'].isna())).sum(),
        'n_served_sensitive': ((df['is_sensitive']) & (df['winner_product_id'].notna())).sum(),
        'mean_clearing_price': round(df.loc[df['winner_product_id'].notna(), 'clearing_price'].mean(), 3),
        'mean_relevance': round(df.loc[df['winner_product_id'].notna(), 'relevance_score'].mean(), 3),
    })
pd.DataFrame(summary)

,variant,n_total,n_served,n_gated_sensitive,n_served_sensitive,mean_clearing_price,mean_relevance
0,defended,50,43,7,3,3.691,0.753
1,revmax,50,50,0,10,3.322,0.738
2,pure_relevance,50,50,0,10,0.000,0.780


**Expected pattern:**
- `defended`: most commercial prompts served, most sensitive prompts gated (welfare reserve does its job).
- `revmax`: most prompts served regardless of sensitivity (no welfare gating).
- `pure_relevance`: every prompt served (no reserve, no auction — picks max-relevance ad).
- Defended mean clearing price ≤ revmax mean clearing price (reserve raises the floor but also gates some auctions entirely).

## 5. Visual demo: render a sponsored answer with image

In [8]:
from IPython.display import HTML, display

# Find a defended record where an ad served and there's an image URL
served = [r for r in all_records['defended'] if r.winner_product_id and r.winner_product_image_url]
if served:
    record = served[0]
    print(f'PROMPT: {record.prompt}\n')
    print(f'CLEAN ANSWER:\n{record.clean_answer}\n')
    print(f'AUCTION: cleared at ${record.clearing_price:.3f} | welfare_loss={record.predicted_welfare_loss:.2f} | relevance={record.relevance_score:.2f}\n')
    display(HTML(mechanism.render_ad_card_html(record)))
else:
    print('No served record with image URL — try a different variant or check image_url coverage.')

PROMPT: I work from home and take 6+ hours of video calls a day. Looking for wireless headphones with great noise cancellation under $400.

CLEAN ANSWER:
For long video-call days, prioritize over-ear headphones with active noise cancellation, dual-device pairing (so you can switch between laptop and phone), and a comfortable headband that doesn't fatigue your head after several hours. In the under-$400 range, the major audio brands all offer flagship ANC models with 25-30 hour battery life and good microphones for calls. Look specifically for transparency mode, multipoint Bluetooth, and a detachable cable for backup. Battery life and call-microphone quality matter more than raw audio fidelity for this use case.

AUCTION: cleared at $1.928 | welfare_loss=0.00 | relevance=0.77



## Handoff notes for Wes and Alex/Swetha

**Wes (gaming + relevance eval):**
```python
import pandas as pd, json
df = pd.read_csv('results/defended.csv')
df['candidate_product_ids'] = df['candidate_product_ids'].apply(json.loads)
# For each row with a winner: attack winner_ad_copy, rerun auction with the modified ad
# Landing-page consistency defense compares modified copy against winner_landing_page
```

**Alex/Swetha (welfare judge + Gradio demo):**
```python
import pandas as pd
df = pd.read_csv('results/defended.csv')
# For Claude welfare judge: triples = list(zip(df['prompt'], df['clean_answer'], df['rewritten_answer']))
# Stratify with df['is_borderline'] to over-sample the hard cases (6 borderline + 4 control)

# For Gradio mocks: import mechanism.render_ad_card_html
# from auction.mechanism import render_ad_card_html, serve_ad
# In Gradio callback: record = serve_ad(prompt, products=..., product_index=..., advertisers=...)
#                      html = render_ad_card_html(record)
```